# System Looting — free public server (Google Colab + playit.gg)

Runs the **System Looting** Luanti server on Google's free Colab machines and
exposes it to the internet through a **playit.gg** tunnel, so anyone can join
from a normal Luanti client. **No credit card needed** — just a Google account.

**How it works:** cell 1 downloads a prebuilt headless Luanti **5.17** server
binary (the distro's minetest-server 5.7 is too old for this game's mods);
cell 2 starts the server; cell 3 hands you a playit.gg claim link for the
public address.

**Limits (free tier):** sessions run up to ~12 h, disconnect after ~90 min of
inactivity, and the VM is wiped afterwards. Great for play sessions, not for a
24/7 server (see `docs/FREE_HOSTING.md` for Oracle free tier etc.).

**How:** Runtime → *Run all*. Then follow the instructions printed at the end
(one-time playit account claim).

In [ ]:
# 1) Engine: prebuilt headless Luanti 5.17 (x86_64, built on Ubuntu 24.04 = Colab)
#    Distro minetest-server on Ubuntu 24.04 is 5.7 and CANNOT run this game
#    (its forked MTG requires ItemStack:add_wear_by_uses, engine >= 5.8).
import subprocess, os, glob

ENG = '/content/engine'
os.makedirs(ENG, exist_ok=True)

if not glob.glob(ENG + '/**/luantiserver', recursive=True):
    print('downloading luantiserver 5.17.0 x86_64 …')
    subprocess.run('curl -sSL -o /content/luantiserver.tar.gz '
                   'https://github.com/rollerozxa/luantiserver/releases/download/5.17.0/'
                   'luantiserver-5.17.0-x86_64.tar.gz',
                   shell=True, check=True, timeout=600)
    subprocess.run(f'tar xzf /content/luantiserver.tar.gz -C {ENG}', shell=True, check=True)

bins = glob.glob(ENG + '/**/luantiserver', recursive=True)
assert bins, 'luantiserver binary not found in the tarball'
engine = bins[0]

v = subprocess.run([engine, '--version'], capture_output=True, text=True)
if v.returncode != 0:
    print('missing shared libs — installing them …')
    subprocess.run('apt-get install -y -qq libsqlite3-0 libzstd1 libgmp10 '
                   'libjsoncpp25 libcurl4 zlib1g', shell=True, timeout=300)
    v = subprocess.run([engine, '--version'], capture_output=True, text=True)
assert v.returncode == 0, v.stderr
print('engine:', engine)
print(v.stdout.splitlines()[0])

In [ ]:
# 2) Game + world + start the server on UDP 30000
import subprocess, shutil, time, os, glob

def find_engine():
    # prebuilt tarball engine
    bins = glob.glob('/content/engine/**/luantiserver', recursive=True)
    if bins:
        return bins[0]
    for c in ('luantiserver', 'minetestserver', 'minetest'):
        p = shutil.which(c)
        if p:
            return p
    for p in ('/usr/games/luantiserver', '/usr/games/minetestserver',
              '/usr/bin/luantiserver', '/usr/bin/minetestserver'):
        if os.path.exists(p):
            return p
    return None

GAME = '/content/SystemTest'   # folder name MUST equal the game id
os.makedirs('/content', exist_ok=True)
if not os.path.isdir(f'{GAME}/.git'):
    subprocess.run('git clone --depth 1 https://github.com/SodoMita/SystemTest '
                   f'{GAME}', shell=True, check=True, timeout=300)
else:
    subprocess.run(f'git -C {GAME} pull --ff-only', shell=True, timeout=120)

engine = find_engine()
assert engine, 'no engine — run cell 1 first'
print('engine:', engine)

# --- make the game visible to the engine ---
# The prebuilt tarball is RUN_IN_PLACE: its user path is <execdir>/.. , so it
# only searches <engine-root>/games (NOT ~/.minetest). Symlink it there.
engine_root = os.path.dirname(os.path.dirname(engine))  # .../luanti
os.makedirs(f'{engine_root}/games', exist_ok=True)
link = f'{engine_root}/games/SystemTest'
os.system(f'rm -rf {link}')
os.symlink(GAME, link)
print('linked:', link, '->', GAME)

# belt & braces: legacy user dirs + env var
for base in (os.path.expanduser('~/.minetest/games'), os.path.expanduser('~/.luanti/games')):
    os.makedirs(base, exist_ok=True)
    l = os.path.join(base, 'SystemTest')
    os.system(f'rm -rf {l}')
    os.symlink(GAME, l)

os.makedirs(f'{GAME}/worlds/systemloot', exist_ok=True)
open(f'{GAME}/worlds/systemloot/world.mt', 'w').write(
    'gameid = SystemTest\nbackend = sqlite3\nmg_name = singlenode\n')
open(f'{GAME}/systemloot.conf', 'w').write('''
server_name = System Looting — Colab
server_description = Free Colab test server
port = 30000
bind_address = 0.0.0.0
max_users = 16
mg_name = singlenode
time_speed = 0
enable_damage = true
sl_auto_start = true
sl_auto_start_delay = 20
''')

open(f'{GAME}/server.log', 'w').close()
open(f'{GAME}/server.err', 'w').close()
env = dict(os.environ, LUANTI_GAME_PATH=GAME)  # 5.17 game path var
proc = subprocess.Popen(
    f'{engine} --gameid SystemTest --world {GAME}/worlds/systemloot '
    f'--config {GAME}/systemloot.conf --logfile {GAME}/server.log --port 30000',
    shell=True, env=env, stdout=subprocess.DEVNULL,
    stderr=open(f'{GAME}/server.err', 'a'))

log = ''
for _ in range(25):
    time.sleep(1)
    try:
        log = open(f'{GAME}/server.log').read()
    except FileNotFoundError:
        continue
    if 'listening on' in log:
        break

if 'listening on' in log:
    print('SERVER UP ✔  port 30000')
    print('\n'.join(l for l in log.splitlines()
                  if 'listening' in l or 'Loaded core' in l or 'arena' in l))
else:
    print('SERVER DID NOT START — process alive:', proc.poll() is None)
    print('--- server.log ---')
    print(log[-2000:] if log else '(empty)')
    print('--- server.err ---')
    print(open(f'{GAME}/server.err').read()[-2000:] or '(empty)')

In [ ]:
# 3) playit.gg tunnel → public address (official headless flow: daemon + playit-cli)
import subprocess, os, time, glob

# 3a) download BOTH binaries from GitHub releases (playit.gg/download serves HTML)
base = 'https://github.com/playit-cloud/playit-agent/releases/download/v1.0.10'
for name in ('playit-linux-amd64', 'playit-cli-linux-amd64'):
    dest = '/content/' + name
    if not os.path.exists(dest) or open(dest,'rb').read(4) != b'\x7fELF':
        subprocess.run(f"curl -sSL -o {dest} '{base}/{name}'", shell=True, check=True, timeout=300)
        assert open(dest,'rb').read(4) == b'\x7fELF', f'{name} download failed'
    os.chmod(dest, 0o755)
print('binaries OK')

# 3b) kill any leftover instances (IPC lock) and start the daemon fresh
subprocess.run('pkill -9 -f /content/playit || true', shell=True)
time.sleep(2)
env = dict(os.environ, TERM='xterm-256color')
p = subprocess.Popen(['/content/playit'], stdout=open('/content/p.log','wb'),
                     stderr=subprocess.STDOUT, env=env)
time.sleep(8)
print('daemon running:', p.poll() is None)

# 3c) generate the claim code + URL via playit-cli (talks to the daemon over IPC)
code = subprocess.run(['/content/playit-cli-linux-amd64', 'claim', 'generate'],
                      capture_output=True, text=True, timeout=60)
print('claim generate rc:', code.returncode)
print(code.stdout.strip()[:300]); print(code.stderr.strip()[:300])
claim_code = (code.stdout or code.stderr).strip().split()[-1] if (code.stdout or code.stderr).strip() else ''
url = subprocess.run(['/content/playit-cli-linux-amd64', 'claim', 'url', claim_code,
                      '--name', 'systemloot-colab'], capture_output=True, text=True, timeout=60)
print('claim url rc:', url.returncode)
print(url.stdout.strip()[:400]); print(url.stderr.strip()[:300])

open('/content/claim_code.txt', 'w').write(claim_code)
print()
print('=' * 66)
print('  CLAIM LINK — open it in your browser (free account, no card):')
print('  ' + (url.stdout or url.stderr).strip())
print('=' * 66)
print('  After you accept it, run the NEXT cell to finish the claim.')
print()
print('daemon log tail:')
print(open('/content/p.log', errors='replace').read()[-1200:])

In [ ]:
# 4) Finish the claim: exchange code → secret → daemon connects
import subprocess, time

claim_code = open('/content/claim_code.txt').read().strip()
print('exchanging claim (waits up to 300 s for you to accept in the browser) …')
ex = subprocess.run(['/content/playit-cli-linux-amd64', 'claim', 'exchange', claim_code,
                     '--wait', '300'], capture_output=True, text=True, timeout=330)
print('exchange rc:', ex.returncode)
print(ex.stdout.strip()[:800]); print(ex.stderr.strip()[:500])

print()
print('Next: in the playit.gg dashboard (playit.gg/account) add a tunnel:')
print('  Protocol UDP, local port 30000   (and one TCP 30000)')
print('Then share the public address playit shows (e.g. 123.45.67.89:51234).')

### After claiming

1. playit.gg dashboard → add tunnel: **UDP**, local port **30000** (and one **TCP** 30000).
2. playit shows a public address like `123.45.67.89:51234` — share it; anyone joins from Luanti.
3. Keep this tab open; interact occasionally (idle >90 min disconnects; sessions end ~12 h — rerun *Run all*; world resets unless backed up, see next cell).

For a permanent free server: `docs/FREE_HOSTING.md` (Oracle Always Free).

In [ ]:
# Optional: back up the world to Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# import subprocess, os
# os.makedirs('/content/drive/MyDrive/systemloot', exist_ok=True)
# subprocess.run('cp -r /content/game/worlds/systemloot /content/drive/MyDrive/systemloot/', shell=True)
# print('world backed up')